# Learning Curve Analysis — Cross-Dataset LOFO

Per-epoch train/val loss & accuracy for `04_cross_dataset_training.py` LOFO runs.

Companion to `learning_curve_analysis.ipynb` (which reads `03_main_training.py`'s
`classification_performances.joblib`, keyed `{exp_title: {mode: {filter: res_entry}}}`).
Cross-dataset results are keyed `{fold_label: {filter: res_entry}}` instead (mode/
curve_type/model are baked into the file path, not dict keys — see `04_output_map.md`),
and live across one file per `(filter, model)` rather than one shared file, so this
notebook loads via `load_partitioned()` instead of a single `joblib.load()`. Everything
downstream (`train_history_{model}_` = list of Keras `History.history` dicts) is the
exact same mechanism `03`/`04` both write through `evaluate_outlier_filters`, so the
plotting helpers below are carried over unchanged from `learning_curve_analysis.ipynb`.

**Note on LOFO's own "folds":** each `fold_label` here is one *held-out chip*, not a
repeated k-fold of the same distribution — so `train_history_{model}_` for a given
(fold_label, filter) is typically a **list of length 1** (one `--train_full`-style
train/test split per fold), not several repeats to average over. Section 7 below
(comparing one model's curve *across* held-out chips) is the more natural "many curves"
view for LOFO than section 4/5's single-fold panel.

In [ ]:
import os, sys
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

_NB_DIR = Path.cwd()
try:
    _nb = globals().get('__vsc_ipynb_file__')
    if _nb:
        _NB_DIR = Path(_nb).resolve().parent
except Exception:
    pass
_ROOT = _NB_DIR.parent.parent  # main_code/
sys.path.insert(0, str(_ROOT))
sys.path.insert(0, str(_ROOT / "utils"))

%load_ext autoreload
%autoreload 2
import config
from cross_dataset_result_io import load_partitioned

%matplotlib inline
print("main_code root:", _ROOT)

## 1 — Point this at the run you want to analyse

In [ ]:
# ── Point this at the group/run whose learning curves you want ─────────────
GROUP_NAME        = "final_4_chip_clean_nn"
EXP_FOLDER        = config.DEFAULT_EXP_FOLDER + "_nc_subtract"
CURVE_TYPE        = "ori_curve_sg_p4_norm"
CURVE_ALIGNMENT   = "pc_ttp"   # main_code's cross_dataset_training.py only supports pc_ttp
PC_TTP_ANCHOR     = "min"
OUTLIER_FILTER    = "noamp_remove"
MODE_STR          = "lofo"

MODELS = ["cnn_gru_dual_attn_recon", "cnn_gru_dual_attn_recon_supcon3",
          "cnn_gru_dual_attn_recon_dann"]
PC_RECENTER_TAG  = "cnn_gru_dual_attn_recon_pc_recentering"
ALL_MODEL_TAGS   = MODELS + [PC_RECENTER_TAG]

FAMILY_LABELS = {
    'cnn_gru_dual': 'CNN-BiGRU',
    'cnn_gru_dual_dann': 'CNN-BiGRU\n+ DANN',
    'cnn_gru_dual_supcon3': 'CNN-BiGRU\n+ Contrastive Loss',
    'cnn_gru_dual_pc_recentering': 'CNN-BiGRU\n+ Latent Alignment',

    'cnn_gru_dual_attn_recon':   'CNN-BiGRU + Spatial Attn',
    'cnn_gru_dual_attn_recon_dann': 'CNN-BiGRU + Spatial Attn\n+ DANN',
    'cnn_gru_dual_attn_recon_supcon3': 'CNN-BiGRU + Spatial Attn\n+ Contrastive Loss',
    'cnn_gru_dual_attn_recon_pc_recentering': 'CNN-BiGRU + Spatial Attn\n+ Latent Alignment',
}

out_dir = Path(EXP_FOLDER) / "cross_dataset_cv" / GROUP_NAME / "curve_alignment_pc_ttp" / f"anchor_{PC_TTP_ANCHOR}"

results = load_partitioned(out_dir, MODE_STR, CURVE_TYPE)
assert results, (f"No '{MODE_STR}' results under {out_dir} for curve_type={CURVE_TYPE}. "
                 f"Check EXP_FOLDER/GROUP_NAME/CURVE_TYPE above.")
print(f"Loaded {len(results)} fold(s) from {out_dir}")

## 2 — Explorer: what history is available?

In [ ]:
for fold_label, fold_res in sorted(results.items()):
    if not isinstance(fold_res, dict):
        continue
    for filter_name, res in fold_res.items():
        if filter_name in ("top_10_features", "class_names") or not isinstance(res, dict):
            continue
        hist_keys = sorted(k for k in res if k.startswith('train_history'))
        if hist_keys:
            models = [k.replace('train_history_', '').rstrip('_') for k in hist_keys]
            print(f"{fold_label} | filter={filter_name}")
            print(f"  models: {models}")
            print()

## 3 — Pick one held-out fold + filter to inspect

In [ ]:
# ── Choose one entry ─────────────────────────────────────────────
TARGET_FOLD   = sorted(k for k in results if k not in ("top_10_features", "class_names"))[0]
TARGET_FILTER = OUTLIER_FILTER   # was hardcoded to None (baseline) -- but no None/baseline
                                  # entry exists on disk for this group, only noamp_remove;
                                  # the cached output below this cell predates that and is stale

res = results[TARGET_FOLD][TARGET_FILTER]
hist_keys = sorted(k for k in res if k.startswith('train_history'))
print(f"Entry : {TARGET_FOLD} | filter={TARGET_FILTER}")
print(f"Models: {[k.replace('train_history_', '').rstrip('_') for k in hist_keys]}")
if hist_keys:
    sample = res[hist_keys[0]][0]
    print(f"Metric keys in first model's history: {list(sample.keys())}")
    print(f"Epochs recorded: {len(sample.get('loss', []))}")

## 4 — Helper functions

Carried over unchanged from `learning_curve_analysis.ipynb`.

In [ ]:
def _get(histories, key):
    # List of np arrays (one per fold-run) for a given metric key
    return [np.array(h[key]) for h in histories if key in h]

def _pad_mean(arrays):
    # Nan-padded mean across arrays of different lengths
    if not arrays:
        return np.array([])
    maxlen = max(len(a) for a in arrays)
    padded = np.full((len(arrays), maxlen), np.nan)
    for i, a in enumerate(arrays):
        padded[i, :len(a)] = a
    return np.nanmean(padded, axis=0)

def _best_epoch(histories, val_key='val_loss'):
    # Epoch index of minimum val_loss per fold-run (where available)
    out = []
    for h in histories:
        if val_key in h:
            out.append(int(np.argmin(h[val_key])))
        else:
            out.append(len(h.get('loss', [])) - 1)
    return out

def _detect_metric_keys(history):
    # Return (loss_key, val_loss_key, acc_key, val_acc_key) for this history dict
    keys = set(history.keys())
    lk  = 'loss'
    vlk = 'val_loss' if 'val_loss' in keys else None
    # accuracy -- standard models use 'accuracy'; MTL/DANN/CORAL models use 'cls_out_accuracy'
    if 'accuracy' in keys:
        ak, vak = 'accuracy', ('val_accuracy' if 'val_accuracy' in keys else None)
    elif 'cls_out_accuracy' in keys:
        ak, vak = 'cls_out_accuracy', ('val_cls_out_accuracy' if 'val_cls_out_accuracy' in keys else None)
    else:
        ak, vak = None, None
    return lk, vlk, ak, vak

BLUE   = 'tab:blue'
ORANGE = 'tab:orange'
ALPHA  = 0.25

def plot_fold_learning_curves(fold_label, filter_name=None):
    """Loss + accuracy learning curves for every model trained on ONE held-out fold.
    This is section 5's plotting body, pulled out into a function so section 10 can
    reuse it per held-out chip instead of duplicating the plotting code."""
    filter_name = OUTLIER_FILTER if filter_name is None else filter_name
    fold_entry = results.get(fold_label, {}).get(filter_name)
    if fold_entry is None:
        print(f"[SKIP] {fold_label}: no results for filter={filter_name}.")
        return
    fold_hist_keys = sorted(k for k in fold_entry if k.startswith('train_history'))
    if not fold_hist_keys:
        print(f"[SKIP] {fold_label}: no train_history under filter={filter_name}.")
        return

    for hist_key in fold_hist_keys:
        model_name = hist_key.replace('train_history_', '').rstrip('_')
        histories  = fold_entry[hist_key]
        if not histories:
            continue

        lk, vlk, ak, vak = _detect_metric_keys(histories[0])
        has_val  = vlk is not None
        has_acc  = ak  is not None

        ncols = 1 + int(has_acc)
        fig, axes = plt.subplots(1, ncols, figsize=(7 * ncols, 4))
        if ncols == 1:
            axes = [axes]
        fig.suptitle(f"{FAMILY_LABELS.get(model_name, model_name)}  |  {fold_label}  ({len(histories)} run(s))",
                    fontsize=12, fontweight='bold')

        ax = axes[0]
        train_l = _get(histories, lk)
        val_l   = _get(histories, vlk) if has_val else []
        for arr in train_l:
            ax.plot(arr, color=BLUE,   alpha=ALPHA, lw=0.8)
        for arr in val_l:
            ax.plot(arr, color=ORANGE, alpha=ALPHA, lw=0.8)
        if train_l:
            ax.plot(_pad_mean(train_l), color=BLUE,   lw=2, label='train')
        if val_l:
            ax.plot(_pad_mean(val_l),   color=ORANGE, lw=2, label='val')
        ax.set_xlabel('Epoch'); ax.set_ylabel('Loss')
        ax.set_title('Loss'); ax.legend(fontsize=8); ax.grid(True, alpha=0.3)

        if has_acc:
            ax = axes[1]
            train_a = _get(histories, ak)
            val_a   = _get(histories, vak) if vak else []
            for arr in train_a:
                ax.plot(arr, color=BLUE,   alpha=ALPHA, lw=0.8)
            for arr in val_a:
                ax.plot(arr, color=ORANGE, alpha=ALPHA, lw=0.8)
            if train_a:
                ax.plot(_pad_mean(train_a), color=BLUE,   lw=2, label='train')
            if val_a:
                ax.plot(_pad_mean(val_a),   color=ORANGE, lw=2, label='val')
            ax.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1.0))
            ax.set_xlabel('Epoch'); ax.set_ylabel('Accuracy')
            ax.set_title('Accuracy'); ax.legend(fontsize=8); ax.grid(True, alpha=0.3)

        plt.tight_layout()
        plt.show()

## 5 — Per-model learning curves (loss + accuracy), this fold

In [ ]:
plot_fold_learning_curves(TARGET_FOLD, TARGET_FILTER)

## 6 — Convergence: how many epochs did each model train, this fold?

In [ ]:
conv_data = {}
for hist_key in hist_keys:
    model_name = hist_key.replace('train_history_', '').rstrip('_')
    histories  = res[hist_key]
    if not histories:
        continue
    epochs_run = [len(h.get('val_loss', h['loss'])) for h in histories]
    conv_data[model_name] = epochs_run

if conv_data:
    fig, ax = plt.subplots(figsize=(max(6, len(conv_data) * 1.2), 4))
    names = list(conv_data.keys())
    vals  = [v[0] for v in conv_data.values()]   # one run per model per fold in LOFO -- no spread to show
    x = np.arange(len(names))
    ax.barh(x, vals, color='steelblue', alpha=0.8)
    ax.set_yticks(x); ax.set_yticklabels([FAMILY_LABELS.get(n, n) for n in names])
    ax.set_xlabel('Epochs trained')
    ax.set_title(f'Convergence -- {TARGET_FOLD}')
    ax.grid(True, axis='x', alpha=0.3)
    plt.tight_layout(); plt.show()

## 7 — Compare one model's learning curve *across held-out chips*

The more natural "many curves" view for LOFO: same model, same filter, one line per
held-out chip -- shows whether convergence/overfitting behaviour is consistent across
folds or driven by one particular chip.

In [ ]:
COMPARE_MODEL  = MODELS[0] if MODELS else (hist_keys[0].replace('train_history_', '').rstrip('_') if hist_keys else None)
COMPARE_METRIC = 'val_loss'
COMPARE_FILTER = OUTLIER_FILTER

fig, ax = plt.subplots(figsize=(10, 5))
found_any = False
for fold_label, fold_res in sorted(results.items()):
    if not isinstance(fold_res, dict) or COMPARE_FILTER not in fold_res:
        continue
    res_e = fold_res[COMPARE_FILTER]
    key = f'train_history_{COMPARE_MODEL}_'
    if key not in res_e:
        continue
    vals = _get(res_e[key], COMPARE_METRIC)
    if not vals:
        continue
    mean_curve = _pad_mean(vals)
    ax.plot(mean_curve, lw=1.5, label=fold_label)
    found_any = True

if found_any:
    ax.set_xlabel('Epoch'); ax.set_ylabel(COMPARE_METRIC)
    ax.set_title(f'{FAMILY_LABELS.get(COMPARE_MODEL, COMPARE_MODEL)} -- {COMPARE_METRIC} across held-out chips (filter={COMPARE_FILTER})')
    ax.legend(fontsize=8, bbox_to_anchor=(1.01, 1), loc='upper left')
    ax.grid(True, alpha=0.3)
    plt.tight_layout(); plt.show()
else:
    print(f"No history found for model '{COMPARE_MODEL}' under filter={COMPARE_FILTER}. "
          f"Check COMPARE_MODEL/COMPARE_FILTER above, or re-run cell 2 to see what's available.")

## 8 — Best val_loss across held-out chips (model comparison)

In [ ]:
best_val = {}
for fold_label, fold_res in sorted(results.items()):
    if not isinstance(fold_res, dict) or COMPARE_FILTER not in fold_res:
        continue
    res_e = fold_res[COMPARE_FILTER]
    for hist_key in (k for k in res_e if k.startswith('train_history')):
        model_name = hist_key.replace('train_history_', '').rstrip('_')
        for h in res_e[hist_key]:
            if 'val_loss' in h:
                best_val.setdefault(model_name, []).append(float(np.min(h['val_loss'])))

if best_val:
    fig, ax = plt.subplots(figsize=(max(6, len(best_val) * 1.2), 4))
    names = list(best_val.keys())
    data  = list(best_val.values())
    bp = ax.boxplot(data, labels=[FAMILY_LABELS.get(n, n) for n in names], patch_artist=True, notch=False)
    for patch in bp['boxes']:
        patch.set_facecolor('steelblue'); patch.set_alpha(0.6)
    ax.set_ylabel('Min val_loss across epochs')
    ax.set_title(f'Best validation loss per model, across held-out chips (filter={COMPARE_FILTER})')
    ax.grid(True, axis='y', alpha=0.3)
    plt.xticks(rotation=30, ha='right')
    plt.tight_layout(); plt.show()
else:
    print(f"No val_loss history found under filter={COMPARE_FILTER}.")

## 9 — Curve confusion grid: (target label + concentration) × predicted label

A confusion-matrix-shaped grid, but each axis and each cell differs from a normal one:

- **Rows** = true `label @ concentration` (not label alone) — so a dataset with 5 labels
  but several concentration levels per label produces more rows than labels (e.g. 5
  labels → 8 label×concentration combinations actually present in a fold).
- **Columns** = predicted label only (concentration isn't predicted).
- **Each cell** is not a count — it's the actual held-out **curves** for every sample
  landing in that (target, prediction) combination (thin lines + bold mean), with `n=`
  in the cell title.

Predictions/true labels come from the cached results (`y_trues_`/`y_preds_{model}_`,
already computed by `04`, cheap to load). Concentration isn't cached anywhere in the
results joblib or the XAI snapshot, so it's reconstructed by re-running the same
pool-building `04` itself uses (`cdt.combine_group`/`combine_group_pc_aligned` +
`cdt.build_lofo_splits`) and slicing to the same `test_idx` — the function then verifies
this reconstruction lines up with the cached `y_trues_` (same fold, same true labels, in
the same row order) before trusting it, so a stale/changed dataset fails loudly instead
of silently mislabelling curves.

One figure per held-out fold (LOFO chip) — `full_data` is skipped, it has no held-out
test set to build a confusion grid from.

In [ ]:
from chip import cross_dataset_training as cdt


def _fmt_conc(v):
    # Mirrors config.apply_well_exclusion's own concentration formatting.
    if v is None:
        return ""
    try:
        return f"{float(v):.0e}"
    except (TypeError, ValueError):
        return str(v)


def plot_curve_confusion_grid(group_name, model_key, curve_type, outlier_filter=None,
                                mode_str="lofo", curve_alignment="pc_ttp",
                                pc_ttp_anchor="min", exp_folder=None,
                                fold_labels=None, max_curves_per_cell=150, seed=0):
    """One figure per held-out LOFO fold: rows = true 'label @ concentration', columns =
    predicted label, each cell = that (target, prediction) combo's held-out curves, titled
    with the sample count and the row-normalized percentage (of all samples with this true
    label+concentration, what % were predicted as this column's label -- standard confusion-
    matrix convention, rows sum to 100%).

    group_name/model_key/curve_type/outlier_filter/mode_str/curve_alignment/pc_ttp_anchor/
    train_center_frac identify one training combination, exactly like the config cell above.
    fold_labels: restrict to specific 'lofo_<chip>' labels (default: every LOFO fold found).
    Returns the list of Figure objects (also displayed inline via plt.show()).
    """
    exp_folder = exp_folder or EXP_FOLDER
    out_dir = Path(exp_folder) / "cross_dataset_cv" / group_name
    if curve_alignment == "pc_ttp":
        out_dir = out_dir / "curve_alignment_pc_ttp" / f"anchor_{pc_ttp_anchor}"

    grid_results = load_partitioned(out_dir, mode_str, curve_type)
    assert grid_results, f"No '{mode_str}' results under {out_dir} for curve_type={curve_type}."

    if model_key not in config.MODEL_KEY_MAP:
        raise ValueError(f"Unknown model_key {model_key!r} -- not in config.MODEL_KEY_MAP.")
    preds_key, _, _ = config.MODEL_KEY_MAP[model_key]

    available_folds = sorted(k for k in grid_results if k.startswith("lofo_"))
    target_folds = [f for f in (fold_labels or available_folds) if f in available_folds]
    if not target_folds:
        print(f"No matching LOFO folds found (available: {available_folds}).")
        return []

    exp_paths = [Path(exp_folder, name) for name in config.CROSS_DATASET_GROUPS[group_name]]

    pc_ttp_cache_dir = None
    combined_shared = None
    if curve_alignment == "pc_ttp":
        pc_ttp_cache_dir = Path(exp_folder) / "cross_dataset_cv" / group_name / "_cache_pc_ttp"
    # main_code's cross_dataset_training.py always aligns via pc_ttp -- no acquisition_start path.

    figs = []
    rng = np.random.RandomState(seed)

    for fold_label in target_folds:
        fold_res = grid_results[fold_label]
        res = fold_res.get(outlier_filter)
        class_names = fold_res.get("class_names")
        if res is None or preds_key not in res or class_names is None:
            print(f"[SKIP] {fold_label}: no cached '{model_key}' predictions "
                  f"(filter={outlier_filter}) or no class_names.")
            continue
        class_names = np.asarray(class_names)

        y_true_cached = np.asarray(res["y_trues_"][0])
        y_pred_cached = np.asarray(res[preds_key][0])
        true_labels = class_names[y_true_cached]
        pred_labels = class_names[y_pred_cached]

        held_out_chip = fold_label.removeprefix("lofo_")
        if curve_alignment == "pc_ttp":
            combined = cdt.combine_group(
                exp_paths, group_name, curve_type, held_out_chip, pc_ttp_cache_dir,
                anchor_method=pc_ttp_anchor, anchor_pct=cdt.PC_TTP_ANCHOR_PCT_DEFAULT)
        else:
            combined = combined_shared
        if combined is None:
            print(f"[SKIP] {fold_label}: could not rebuild the pool for this fold.")
            continue

        lofo_splits = cdt.build_lofo_splits(combined["dataset_id"])
        if fold_label not in lofo_splits:
            print(f"[SKIP] {fold_label}: not present in the reconstructed LOFO splits "
                  f"(found: {sorted(lofo_splits)}).")
            continue
        _, test_idx = lofo_splits[fold_label]

        recon_true_labels = np.asarray(combined["Y_mapped"])[test_idx]
        if recon_true_labels.shape != true_labels.shape or not np.array_equal(recon_true_labels, true_labels):
            print(f"[SKIP] {fold_label}: reconstructed test split doesn't match the cached "
                  f"y_trues_ -- dataset may have changed since training. Skipping to avoid "
                  f"mislabelled curves.")
            continue

        curves_test = combined["curves"][test_idx]
        conc_test   = combined["concentration_raw"][test_idx]
        conc_str    = np.array([_fmt_conc(c) for c in conc_test])
        target_combo = np.array([f"{lbl} @ {c}" if c else lbl
                                 for lbl, c in zip(true_labels, conc_str)])

        target_cats = sorted(set(target_combo))
        pred_cats   = list(class_names)
        row_totals  = {tcat: int((target_combo == tcat).sum()) for tcat in target_cats}

        fig, axes = plt.subplots(len(target_cats), len(pred_cats),
                                 figsize=(2.2 * len(pred_cats), 1.8 * len(target_cats)),
                                 squeeze=False)
        for i, tcat in enumerate(target_cats):
            row_n = row_totals[tcat]
            for j, pcat in enumerate(pred_cats):
                ax = axes[i, j]
                mask = (target_combo == tcat) & (pred_labels == pcat)
                n = int(mask.sum())
                pct = (n / row_n * 100) if row_n else 0.0
                sub_curves = curves_test[mask]
                if n > max_curves_per_cell:
                    idx = rng.choice(n, max_curves_per_cell, replace=False)
                    sub_curves = sub_curves[idx]
                for c in sub_curves:
                    ax.plot(c, color='tab:blue', alpha=0.15, lw=0.6)
                if n:
                    ax.plot(curves_test[mask].mean(axis=0), color='tab:red', lw=1.4)
                ax.set_xticks([]); ax.set_yticks([])
                ax.set_title(f"n={n} ({pct:.2f}%)", fontsize=7)
                if j == 0:
                    ax.set_ylabel(f"{tcat}\n(n={row_n})", fontsize=7)
                if i == 0:
                    ax.annotate(f"pred: {pcat}", xy=(0.5, 1.5), xycoords='axes fraction',
                               ha='center', fontsize=8, fontweight='bold')

        fig.suptitle(f"{group_name} | {FAMILY_LABELS.get(model_key, model_key)} | {curve_type} | "
                    f"filter={outlier_filter} | {fold_label}", fontsize=11, fontweight='bold', y=1.03)
        fig.tight_layout()
        figs.append(fig)
        plt.show()

    return figs

In [ ]:
%matplotlib inline

In [ ]:
# ── Example call -- adjust to a (group, model, curve_type, ...) combination you've trained ──
%matplotlib inline
_ = plot_curve_confusion_grid(
    group_name=GROUP_NAME,
    model_key=MODELS[0],
    curve_type=CURVE_TYPE,
    outlier_filter=OUTLIER_FILTER,
    mode_str=MODE_STR,
    curve_alignment=CURVE_ALIGNMENT,
    pc_ttp_anchor=PC_TTP_ANCHOR,
)

## 10 — Per-model learning curves, every held-out fold

Section 5's panel (loss + accuracy per model), looped across **every** held-out chip
instead of just the one picked in section 3 -- one set of plots per fold, so you can
scan all of them without re-picking `TARGET_FOLD` each time. `full_data` is skipped
(no held-out test set, so `val_*` metrics aren't meaningful the same way).

In [ ]:
ALL_LOFO_FOLDS = sorted(k for k in results if k.startswith("lofo_"))
print(f"{len(ALL_LOFO_FOLDS)} held-out fold(s): {ALL_LOFO_FOLDS}")

for fold_label in ALL_LOFO_FOLDS:
    print(f"\n{'='*90}\n{fold_label}\n{'='*90}")
    plot_fold_learning_curves(fold_label, OUTLIER_FILTER)